In [0]:
import os
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F



# ==========================================
# NATIVE PYSPARK LOGGING CONFIGURATION
# ==========================================
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("NYCTaxiMedallionPipeline")

# Setup paths
BASE_PATH = "/Volumes/dev_nyc"
RAW_CSV   = f"{BASE_PATH}/landing/nyc_taxi_trip_duration/generated_trips_raw.csv"
BRONZE    = f"{BASE_PATH}/bronze/taxi_duartion"
SILVER    = f"{BASE_PATH}/silver/taxi_duartion"
GOLD      = f"{BASE_PATH}/gold/taxi_duration"

# ==========================================
# ENSURE VOLUMES EXIST
# ==========================================
spark.sql("CREATE VOLUME IF NOT EXISTS dev_nyc.bronze.taxi_duartion")
spark.sql("CREATE VOLUME IF NOT EXISTS dev_nyc.silver.taxi_duartion")
spark.sql("CREATE VOLUME IF NOT EXISTS dev_nyc.gold.taxi_duration")

# ==========================================
# RUN THE PIPELINE STEPS
# ==========================================
try:
    logger.info("--- STARTING PYSPARK PIPELINE RUN ---")

    # 1. BRONZE STEP: Read raw CSV and append ingestion time
    logger.info("Step 1: Running Bronze Layer...")
    df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(RAW_CSV)

    df_bronze = df_raw.withColumn("_ingested_at", F.current_timestamp())
    df_bronze.columns  # Force analysis on Spark Connect
    df_bronze.write.mode("overwrite").parquet(BRONZE)

    bronze_count = df_bronze.count()
    logger.info(f"Bronze complete. Saved {bronze_count} raw records.")

    # 2. SILVER STEP: Clean, cast timestamps, and calculate duration
    logger.info("Step 2: Running Silver Layer...")
    df_silver_src = spark.read.parquet(BRONZE)

    # Cast dates and apply business rules
    df_clean = df_silver_src \
        .withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime")) \
        .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime")) \
        .filter((F.col("passenger_count") > 0) & (F.col("fare_amount") > 0))

    # Calculate duration in minutes: (dropoff - pickup) in seconds / 60
    df_silver = df_clean.withColumn(
        "trip_duration_minutes",
        F.round((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 2)
    )
    df_silver.columns  # Force analysis on Spark Connect
    df_silver.write.mode("overwrite").parquet(SILVER)

    silver_count = df_silver.count()
    logger.info(f"Silver complete. Kept {silver_count} clean records (Dropped {bronze_count - silver_count}).")

    # 3. GOLD STEP: Daily business aggregation
    logger.info("Step 3: Running Gold Layer...")
    df_gold_src = spark.read.parquet(SILVER)

    # Aggregate data by pickup date
    df_gold = df_gold_src \
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
        .groupBy("pickup_date") \
        .agg(
            F.count("VendorID").alias("total_trips"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.mean("trip_duration_minutes"), 2).alias("avg_duration")
        ) \
        .orderBy("pickup_date")
    df_gold.columns  # Force analysis on Spark Connect
    df_gold.write.mode("overwrite").parquet(f"{GOLD}/gold_daily_summary.parquet")

    gold_count = df_gold.count()
    logger.info(f"Gold complete. Saved summary rows for {gold_count} days.")
    logger.info("--- PYSPARK PIPELINE FINISHED SUCCESSFULLY ---")

except Exception as e:
    logger.error(f"!!! PYSPARK PIPELINE FAILED: {str(e)}")
    raise e
